# DriveSeg: Autonomous Road Scene Segmentation

**arXiv ref:** *Comparative performance analysis of U-Net and DeepLabV3+ for semantic segmentation in traffic environments* — Scientific Reports 2026

This notebook demonstrates pixel-level road segmentation using **DeepLabV3+** with a ResNet-101 backbone pretrained on Cityscapes.


In [ ]:
from __future__ import annotations
import sys, os; sys.path.insert(0, os.path.abspath('.'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import make_synthetic, create_dataloaders, CLASS_NAMES, NUM_CLASSES
from src.model import build_deeplab
from src.core import compute_seg_metrics
from src.visualizations import plot_sample, plot_metrics_radar, _style


In [ ]:
# Generate synthetic road scene data
n = 200
data = make_synthetic(n=n, seed=42)
print(f"Generated {data['n_samples']} samples")
print(f"Image shape: {data['images'].shape}")
print(f"Classes: {data['class_names']}")


# Show sample data
fig = plot_sample(data['images'][0], data['masks'][0])
plt.show()


In [ ]:
# Build DeepLabV3+ model
model = build_deeplab(num_classes=NUM_CLASSES, pretrained_backbone=False)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Architecture: DeepLabV3+ (ResNet-101 backbone + ASPP + decoder)")


In [ ]:
# Forward pass on a batch
with torch.no_grad():
    batch_x = data['images'][:4]
    out = model(batch_x)['out']
    preds = out.argmax(dim=1)
print(f"Output shape: {out.shape}")
print(f"Predictions shape: {preds.shape}")


In [ ]:
# Compute segmentation metrics
metrics = compute_seg_metrics(preds.numpy(), data['masks'][:4].numpy(), NUM_CLASSES)
print(f"mIoU: {metrics['miou']:.4f}")
print(f"Dice:  {metrics['dice']:.4f}")
print(f"Pixel Acc: {metrics['pixel_acc']:.4f}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:12s} IoU={metrics['per_class_iou'][i]:.4f}  Dice={metrics['per_class_dice'][i]:.4f}")


# Visualize
fig = plot_metrics_radar(metrics)
plt.show()


In [ ]:
print('DriveSeg notebook complete. DeepLabV3+ performs pixel-level segmentation with ASPP multi-scale context fusion.')
